In [5]:
import os
import sys
import yaml

# 1. 프로젝트 최상단 루트 폴더로 현재 작업 디렉토리 맞추기
current_dir = os.getcwd()
if current_dir.endswith("notebook"):
    os.chdir("..")  # notebook 안에서 실행중이면 상위(프로젝트 루트)로 이동

# 2. 파이썬 sys.path에도 루트 추가
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 3. 이제 프로젝트 루트 기준이므로 정상적으로 파일을 읽습니다!
with open("config/default.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("설정 로드 완료:", config["generation"])

설정 로드 완료: {'provider': 'openai', 'model': 'gpt-5-nano', 'temperature': 0.1, 'top_p': 0.95, 'max_tokens': 1000}


In [6]:
import json
from src.retriever import SearchResult

# 2. PM님이 만들어둔 가상 청크 데이터(JSONL)를 실전처럼 불러오기
mock_results = []
file_path = "samples/processed/sample_chunks.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        # JSON 데이터를 SearchResult 객체로 변환
        result = SearchResult(
            chunk_id=data["chunk_id"],
            doc_id=data["doc_id"],
            text=data["text"],
            score=0.85,  # 가상의 검색 유사도 점수
            metadata=data["metadata"]
        )
        mock_results.append(result)

print(f"총 {len(mock_results)}개의 실전용 청크 데이터를 성공적으로 불러왔습니다!")

총 3개의 실전용 청크 데이터를 성공적으로 불러왔습니다!


In [7]:
# 3. 문서 내용에 기반한 질문 테스트
from src.rag_engine import generate_answer
question_1 = "기존 시스템에서 사용자 정보 이전할 때 제외해야 하는 항목이 뭐야? 그리고 개인정보 다운로드 기록은 얼마나 보관해야 해?"
response_1 = generate_answer(question_1, mock_results, config)

print(f"Q: {question_1}\n")
print(f"A:\n{response_1['answer']}")

Q: 총사업예산과 수행 기간은 어떻게 돼?

A:
총사업예산은 480,000,000원(부가가치세 포함)이고, 수행 기간은 계약 체결일로부터 8개월입니다. (참고: 본 문서는 가상 데이터로 실제 기관/사업과 무관합니다.)


In [4]:
# 4. 문서에 없는 엉뚱한 질문을 했을 때 추측하지 않고 차단하는지 테스트
question_2 = "이 플랫폼에 블록체인 연계 기능이 포함되어 있어?"
response_2 = generate_answer(question_2, mock_results, config)

print(f"Q: {question_2}\n")
print(f"A:\n{response_2['answer']}")

Q: 이 플랫폼에 블록체인 연계 기능이 포함되어 있어?

A:
제공된 문서에서 블록체인 연계 기능의 포함 여부에 대한 근거를 찾을 수 없어 확인할 수 없습니다. 문서는 총사업예산과 수행 기간만 명시하고 있으며, 블록체인 관련 구체적 기능 정보는 제공되지 않았습니다. 추가 문서나 RFP의 상세 스코프를 확인해 주시면 더 도와드리겠습니다.
